# MCP Server Lab
Build and test a real MCP Server in Python.

**No API key needed** — we test the server directly with a Python MCP Client.

> Connection to tool calling lab:
> - Tool calling: `tool_map["calculate"](**args)` — direct function call
> - MCP Server: `session.call_tool("calculate", args)` — via protocol

In [ ]:
!pip install -q mcp

## Define the MCP Server

Same functions from the tool calling lab — just add `@mcp.tool()`.

In [ ]:
server_code = '''
from mcp.server.fastmcp import FastMCP
from datetime import date

mcp = FastMCP("My Server")

@mcp.tool()
def calculate(expression: str) -> str:
    """Evaluate a math expression. Example: 240 * 0.15"""
    return str(eval(expression))

@mcp.tool()
def get_today() -> str:
    """Get today date in YYYY-MM-DD format."""
    return str(date.today())

mcp.run()
'''

with open('my_server.py', 'w') as f:
    f.write(server_code)

print('my_server.py saved.')

## Connect MCP Client & List Tools

The client launches `my_server.py` as a subprocess and connects via stdio.
`list_tools()` asks: *what tools do you have?*

In [ ]:
import sys
sys.stderr = sys.__stderr__  # Colab fix: restore real stderr so MCP subprocess can use fileno()

import asyncio
from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client

server_params = StdioServerParameters(command='python', args=['my_server.py'])

async def list_tools():
    async with stdio_client(server_params) as (read, write):
        async with ClientSession(read, write) as session:
            await session.initialize()
            result = await session.list_tools()
            print('Tools available:')
            for tool in result.tools:
                print(f'  - {tool.name}: {tool.description}')

await list_tools()

## Call a Tool

`call_tool()` sends the request via MCP protocol → server executes → returns result.

In [ ]:
import sys
sys.stderr = sys.__stderr__  # Colab fix

async def call_tools():
    async with stdio_client(server_params) as (read, write):
        async with ClientSession(read, write) as session:
            await session.initialize()

            # Call calculate
            r1 = await session.call_tool('calculate', {'expression': '240 * 0.15'})
            print(f"calculate('240 * 0.15') -> {r1.content[0].text}")

            # Call get_today
            r2 = await session.call_tool('get_today', {})
            print(f"get_today() -> {r2.content[0].text}")

await call_tools()

## Free Play — Add Your Own Tool

Edit `server_code` in Cell 2, re-run it, then call your new tool here.

In [ ]:
import sys
sys.stderr = sys.__stderr__  # Colab fix

# Example: add this to server_code in Cell 2
#
# @mcp.tool()
# def celsius_to_fahrenheit(celsius: float) -> str:
#     """Convert Celsius to Fahrenheit"""
#     return str(celsius * 9/5 + 32)

async def freeplay():
    async with stdio_client(server_params) as (read, write):
        async with ClientSession(read, write) as session:
            await session.initialize()
            tools = await session.list_tools()
            print('Available tools:', [t.name for t in tools.tools])
            # call your tool here:
            # r = await session.call_tool('your_tool', {'arg': 'value'})

await freeplay()